# 12. Breeden & Crook (2022) Multihorizon Discrete-Time Survival Model

## Overview

This notebook implements the multihorizon discrete-time survival model from **Breeden & Crook (2022)** for competing risks (prepayment and default) on the Freddie Mac dataset.

### Key Innovation

Rather than fitting a single model, Breeden & Crook fit **separate logistic regressions for each forecast horizon** $L = 1, \ldots, 12$, where model $L$ uses delinquency lagged by $L$ months. This means:
- **Short horizons**: Delinquency dominates predictions (strong roll-rate/state-transition effect)
- **Long horizons**: Origination variables (LTV, DTI, FICO) take over as delinquency's predictive power decays

### Model Components

| Component | Description | Variables |
|-----------|-------------|----------|
| Lifecycle $F(a)$ | Maturation pattern | B-spline basis on loan age (5 knots) |
| Vintage $G(v)$ | Origination quality | Year dummies (2010-2025) |
| Environment $H(t)$ | Calendar-time effects | 12 macro variables |
| Static origination | Credit quality at origination | FICO, DTI, LTV, interest rate, log(UPB) |
| Behavioral | Dynamic performance | `bal_repaid_lag1`, `t_act_12m`, rolling delinquency counts |
| **Delinquency (lag $L$)** | **Key Breeden-Crook feature** | $D_{1m}^{lag L}$, $D_{2m}^{lag L}$, ..., $D_{5m+}^{lag L}$ |

### Competing Risks

We adapt the model for two competing events:
- **$k = 1$**: Prepayment
- **$k = 2$**: Default (90+ DPD)

Total models: 2 risks $\times$ (12 horizons + 1 origination) = **26 logistic regressions**

### Reference

Breeden, J.L. and Crook, J.N. (2022). "Multihorizon discrete time survival models." *Journal of the Operational Research Society*.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from src.competing_risks.breeden_crook import (
    BreedenCrookMultihorizon,
    enrich_panel_with_delinquency,
    create_delinquency_indicators,
    create_lagged_delinquency,
    plot_delinquency_coefficients,
    plot_origination_coefficients,
    plot_pseudo_r2_by_horizon,
    DELINQ_INDICATOR_COLS,
    EVENT_NAMES,
)
from src.competing_risks.evaluation import (
    time_dependent_concordance_index,
    evaluate_all_events,
    brier_score_competing_risks,
    calibration_plot,
    EVAL_TIMES,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

print('Imports successful')

## 1. Configuration

In [ ]:
# Paths
DATA_DIR = Path('../data/processed')
RAW_DIR = Path('../data/raw')
FIGURES_DIR = Path('../reports/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Cross-validation folds
TRAIN_FOLDS = list(range(9))  # Folds 0-8
VAL_FOLDS = [9]               # Fold 9
TEST_FOLD = 10                 # Fold 10

# Model configuration
MAX_HORIZON = 12    # Forecast horizons L=1,...,12
CIF_HORIZON = 72    # Maximum CIF prediction horizon
N_AGE_KNOTS = 5     # Spline knots for lifecycle F(a)
SEED = 42

np.random.seed(SEED)

## 2. Data Preparation

### 2.1 Load panel and enrich with delinquency status

The existing `loan_month_panel.parquet` has rolling 12-month delinquency counts but not the month-by-month `current_loan_delinquency_status` (0/1/2/3+) needed for the Breeden-Crook lagged indicators. We extract this from the raw performance files.

In [ ]:
# Load panel
panel_df = pd.read_parquet(DATA_DIR / 'loan_month_panel.parquet')
print(f'Panel: {len(panel_df):,} rows, {panel_df["loan_sequence_number"].nunique():,} loans')
print(f'Columns: {list(panel_df.columns)}')

In [ ]:
# Enrich with delinquency status from raw performance files
# This is cached to avoid re-reading raw files each time
cache_path = DATA_DIR / 'loan_month_panel_with_delinq.parquet'
panel_df = enrich_panel_with_delinquency(panel_df, RAW_DIR, cache_path=cache_path)

print(f'\nDelinquency status distribution:')
print(panel_df['delinq_status'].value_counts().sort_index())

### 2.2 Create delinquency indicators and lagged features

In [ ]:
# Create binary delinquency state indicators
panel_df = create_delinquency_indicators(panel_df)

print('Delinquency indicator means:')
for col in DELINQ_INDICATOR_COLS:
    print(f'  {col}: {panel_df[col].mean():.4%}')

In [ ]:
# Create lagged delinquency indicators for each horizon L=1,...,12
panel_df = create_lagged_delinquency(panel_df, max_lag=MAX_HORIZON)

print(f'Panel shape after enrichment: {panel_df.shape}')
print(f'New columns (sample): {[c for c in panel_df.columns if "lag" in c][:10]}')

In [ ]:
# Create additional features
if 'orig_upb' in panel_df.columns:
    panel_df['log_orig_upb'] = np.log(panel_df['orig_upb'].astype(float).clip(lower=1))

if 'bal_repaid' in panel_df.columns:
    panel_df['bal_repaid_lag1'] = panel_df.groupby('loan_sequence_number')['bal_repaid'].shift(1)

### 2.3 Train / Validation / Test split

In [ ]:
train_panel = panel_df[panel_df['fold'].isin(TRAIN_FOLDS)].copy()
val_panel = panel_df[panel_df['fold'].isin(VAL_FOLDS)].copy()
test_panel = panel_df[panel_df['fold'] == TEST_FOLD].copy()

print(f'Train (folds {TRAIN_FOLDS}): {len(train_panel):,} rows, '
      f'{train_panel["loan_sequence_number"].nunique():,} loans')
print(f'Val (fold {VAL_FOLDS}): {len(val_panel):,} rows, '
      f'{val_panel["loan_sequence_number"].nunique():,} loans')
print(f'Test (fold {TEST_FOLD}): {len(test_panel):,} rows, '
      f'{test_panel["loan_sequence_number"].nunique():,} loans')

# Event distribution
print('\nEvent distribution (train terminal observations):')
train_terminal = train_panel.groupby('loan_sequence_number').last().reset_index()
for code in sorted(train_terminal['event_code'].unique()):
    count = (train_terminal['event_code'] == code).sum()
    print(f'  {EVENT_NAMES.get(code, "Other")} (k={code}): {count:,}')

## 3. APC Exploration

Before fitting the multihorizon model, we explore the Age-Period-Cohort (APC) structure.
- **Age $F(a)$**: Empirical hazard by loan age (maturation pattern)
- **Vintage $G(v)$**: Hazard by origination year (cohort quality)
- **Calendar time $H(t)$**: Hazard by calendar time (economic environment)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for row_idx, (event_code, event_name) in enumerate([(2, 'Default'), (1, 'Prepay')]):
    # Age effect F(a)
    age_hazard = train_panel.groupby('loan_age').apply(
        lambda g: (g['event_code'] == event_code).mean()
    )
    axes[row_idx, 0].plot(age_hazard.index, age_hazard.values, linewidth=1.5)
    axes[row_idx, 0].set_xlabel('Loan Age (months)')
    axes[row_idx, 0].set_ylabel('Empirical Hazard Rate')
    axes[row_idx, 0].set_title(f'{event_name}: Age Effect F(a)')
    
    # Vintage effect G(v)
    vintage_hazard = train_panel.groupby('vintage_year').apply(
        lambda g: (g['event_code'] == event_code).mean()
    )
    axes[row_idx, 1].bar(vintage_hazard.index, vintage_hazard.values, alpha=0.7)
    axes[row_idx, 1].set_xlabel('Vintage Year')
    axes[row_idx, 1].set_ylabel('Empirical Hazard Rate')
    axes[row_idx, 1].set_title(f'{event_name}: Vintage Effect G(v)')
    axes[row_idx, 1].tick_params(axis='x', rotation=45)
    
    # Calendar time effect H(t)
    if 'year_month' in train_panel.columns:
        cal_hazard = train_panel.groupby('year_month').apply(
            lambda g: (g['event_code'] == event_code).mean()
        )
        axes[row_idx, 2].plot(cal_hazard.index.astype(str), cal_hazard.values, linewidth=1)
        axes[row_idx, 2].set_xlabel('Calendar Time')
        axes[row_idx, 2].set_ylabel('Empirical Hazard Rate')
        axes[row_idx, 2].set_title(f'{event_name}: Calendar Time Effect H(t)')
        # Show fewer x-ticks for readability
        n_ticks = len(cal_hazard)
        step = max(1, n_ticks // 10)
        axes[row_idx, 2].set_xticks(axes[row_idx, 2].get_xticks()[::step])
        axes[row_idx, 2].tick_params(axis='x', rotation=45)

plt.suptitle('Age-Period-Cohort (APC) Structure', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'breeden_crook_apc_exploration.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Multihorizon Model Training

Fit 26 logistic regressions:
- **Origination model** (horizon 0): No delinquency features, for loans < 6 months old
- **Horizon models** (L=1,...,12): Include `D_Xm_lagL` delinquency indicators

For each horizon, regularization strength $C$ is tuned on the validation fold.

In [ ]:
model = BreedenCrookMultihorizon(
    max_horizon=MAX_HORIZON,
    C_values=[0.01, 0.1, 1.0, 10.0],
    solver='lbfgs',
    n_age_knots=N_AGE_KNOTS,
    seed=SEED,
)

model.fit(train_panel, val_panel)

In [ ]:
# Validation performance by horizon
r2_df = model.get_pseudo_r2()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, ylabel in zip(axes, ['auc_val', 'log_loss_val'], 
                               ['Validation AUC', 'Validation Log-Loss']):
    for risk in ['default', 'prepay']:
        data = r2_df[(r2_df['risk'] == risk) & (r2_df['horizon'] > 0)].sort_values('horizon')
        ax.plot(data['horizon'], data[metric], 'o-', label=risk.title(), 
                linewidth=2, markersize=6)
    ax.set_xlabel('Forecast Horizon L')
    ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} by Horizon')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(range(1, MAX_HORIZON + 1))

plt.tight_layout()
plt.show()

# Print validation AUC table
print('Validation AUC by Horizon:')
print(r2_df.pivot(index='horizon', columns='risk', values='auc_val').round(4))

## 5. Coefficient Analysis Across Horizons

This is the **key section** that reproduces the paper's central findings:
1. Delinquency coefficients decay with forecast horizon
2. Origination variable coefficients increase and stabilize
3. Model discriminatory power (Gini) decreases with horizon

In [ ]:
coef_df = model.get_coefficients()
print(f'Total coefficients: {len(coef_df):,}')
coef_df.head(10)

### 5.1 Delinquency Coefficients vs. Horizon (Paper Fig. 5)

Shows how delinquency's predictive power **decays** with forecast horizon. At L=1, a 3-month delinquent loan has a very high default probability; at L=12, the same delinquency status is much less predictive.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_delinquency_coefficients(coef_df, risk='default', ax=axes[0])
plot_delinquency_coefficients(coef_df, risk='prepay', ax=axes[1])

plt.suptitle('Delinquency Coefficients vs. Forecast Horizon (Breeden & Crook Fig. 5)', 
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'breeden_crook_coef_delinquency.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Origination Variable Coefficients vs. Horizon (Paper Fig. 6)

Shows how origination variables' predictive power **increases** with forecast horizon as delinquency loses its dominance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_origination_coefficients(coef_df, risk='default', ax=axes[0])
plot_origination_coefficients(coef_df, risk='prepay', ax=axes[1])

plt.suptitle('Origination Variable Coefficients vs. Horizon (Breeden & Crook Fig. 6)', 
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'breeden_crook_coef_origination.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Model Fit by Horizon (Paper Figs. 8-9)

Gini coefficient (2*AUC - 1) shows model discriminatory power declining with horizon, as near-future events are easier to predict.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_pseudo_r2_by_horizon(r2_df, ax=ax)
plt.suptitle('Model Discriminatory Power by Horizon (Breeden & Crook Figs. 8-9)', 
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'breeden_crook_pseudo_r2.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. CIF Prediction & Evaluation

### 6.1 Generate CIF predictions

For each test loan, use the terminal observation as the forecast origin. All horizon-$L$ models use delinquency at the forecast origin $t_0$, because model $L$ was trained with lag $L$ and predicts at $t_0 + L$.

In [ ]:
# Get terminal observations for test loans
test_terminal = test_panel.groupby('loan_sequence_number').last().reset_index()
print(f'Test loans: {len(test_terminal):,}')

# Generate CIF
CIF_def, CIF_pre, S = model.predict_cif(test_terminal, max_months=CIF_HORIZON)

# Validity check: CIF_def + CIF_pre + S = 1
check = CIF_def + CIF_pre + S
print(f'\nCIF validity (CIF_def + CIF_pre + S):')
print(f'  Min: {check.min():.6f}, Max: {check.max():.6f}, Mean: {check.mean():.6f}')

### 6.2 Time-dependent C-index

In [ ]:
event_times = test_terminal['loan_age'].values.astype(float)
event_codes = test_terminal['event_code'].values.astype(int)

# Evaluate at standard horizons
results_rows = []
for tau in EVAL_TIMES:
    tau_idx = min(tau, CIF_HORIZON)
    risk_prepay = CIF_pre[:, tau_idx]
    risk_default = CIF_def[:, tau_idx]

    c_prepay, _, _ = time_dependent_concordance_index(
        event_times, event_codes, risk_prepay, tau, event_of_interest=1)
    c_default, _, _ = time_dependent_concordance_index(
        event_times, event_codes, risk_default, tau, event_of_interest=2)

    results_rows.append({
        'Metric': f'C({tau})',
        'Prepay (k=1)': c_prepay,
        'Default (k=2)': c_default,
    })
    print(f'C({tau}): Prepay = {c_prepay:.4f}, Default = {c_default:.4f}')

results_df = pd.DataFrame(results_rows)
mean_prepay = results_df['Prepay (k=1)'].mean()
mean_default = results_df['Default (k=2)'].mean()
results_df = pd.concat([results_df, pd.DataFrame([{
    'Metric': 'mean_C',
    'Prepay (k=1)': mean_prepay,
    'Default (k=2)': mean_default,
}])], ignore_index=True)
results_df['Combined'] = (results_df['Prepay (k=1)'] + results_df['Default (k=2)']) / 2

print(f'\nMean C-index: Prepay={mean_prepay:.4f}, Default={mean_default:.4f}, '
      f'Combined={(mean_prepay + mean_default)/2:.4f}')

results_df

### 6.3 Brier Scores

In [ ]:
print('Brier Scores:')
brier_rows = []
for tau in EVAL_TIMES:
    tau_idx = min(tau, CIF_HORIZON)
    bs_prepay = brier_score_competing_risks(
        event_times, event_codes, CIF_pre[:, tau_idx], tau, event_of_interest=1)
    bs_default = brier_score_competing_risks(
        event_times, event_codes, CIF_def[:, tau_idx], tau, event_of_interest=2)
    brier_rows.append({'Horizon': tau, 'BS Prepay': bs_prepay, 'BS Default': bs_default})
    print(f'  BS({tau}): Prepay = {bs_prepay:.6f}, Default = {bs_default:.6f}')

pd.DataFrame(brier_rows)

### 6.4 Comparison with other models

Load results from other model notebooks for comparison.

In [ ]:
# Load previous results if available
comparison_models = {}
results_dir = Path('../results')

for model_file, model_name in [
    ('deephit_cindex.csv', 'DeepHit'),
    ('fine_gray_cindex.csv', 'Fine-Gray'),
    ('cox_cindex.csv', 'Cause-Specific Cox'),
    ('dynamic_deephit_cindex.csv', 'Dynamic-DeepHit'),
]:
    fpath = results_dir / model_file
    if fpath.exists():
        comparison_models[model_name] = pd.read_csv(fpath)
        print(f'Loaded {model_name} results')

# Display Breeden-Crook results alongside others
print('\n' + '=' * 60)
print('BREEDEN-CROOK MULTIHORIZON RESULTS')
print('=' * 60)
print(results_df.to_string(index=False))

## 7. Diagnostic Plots

### 7.1 Sample CIF Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

np.random.seed(42)
sample_idx = np.random.choice(len(CIF_def), size=5, replace=False)
months = np.arange(CIF_def.shape[1])

for ax, data, title, ylabel in zip(axes,
    [CIF_def[sample_idx], CIF_pre[sample_idx], S[sample_idx]],
    ['Default CIF', 'Prepayment CIF', 'Survival S(t)'],
    ['Cumulative Incidence', 'Cumulative Incidence', 'Survival Probability'],
):
    for i, idx in enumerate(sample_idx):
        event_str = EVENT_NAMES.get(int(event_codes[idx]), '?')
        ax.plot(months, data[i], label=f'Loan {idx} ({event_str})', alpha=0.7)
    ax.set_xlabel('Months from Forecast Origin')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1)

plt.suptitle('Breeden-Crook: Sample CIF Curves', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'breeden_crook_cif_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.2 Calibration Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, event_code, event_name, cif_arr in zip(
    axes, [1, 2], ['Prepayment', 'Default'], [CIF_pre, CIF_def]
):
    # Use tau=48 for calibration
    tau = 48
    tau_idx = min(tau, CIF_HORIZON)
    observed = ((event_times <= tau) & (event_codes == event_code)).astype(float)
    predicted = cif_arr[:, tau_idx]
    calibration_plot(observed, predicted, ax=ax,
                     title=f'{event_name} Calibration (tau={tau})')

plt.tight_layout()
plt.show()

### 7.3 Average CIF by Horizon

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

months = np.arange(CIF_HORIZON + 1)
ax.plot(months, CIF_def.mean(axis=0), 'r-', linewidth=2, label='Default CIF (mean)')
ax.plot(months, CIF_pre.mean(axis=0), 'b-', linewidth=2, label='Prepay CIF (mean)')
ax.plot(months, S.mean(axis=0), 'g-', linewidth=2, label='Survival S(t) (mean)')

# Add confidence band
ax.fill_between(months, 
                np.percentile(CIF_def, 25, axis=0),
                np.percentile(CIF_def, 75, axis=0), 
                alpha=0.15, color='red')
ax.fill_between(months,
                np.percentile(CIF_pre, 25, axis=0),
                np.percentile(CIF_pre, 75, axis=0),
                alpha=0.15, color='blue')

ax.set_xlabel('Months from Forecast Origin')
ax.set_ylabel('Probability')
ax.set_title('Breeden-Crook: Average CIF and Survival (Test Set)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

### 7.4 C-index Comparison Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

cindex_data = results_df[results_df['Metric'].str.startswith('C(')]
horizons = [24, 48, 72]
x = np.arange(len(horizons))
width = 0.35

prepay_vals = cindex_data['Prepay (k=1)'].values
default_vals = cindex_data['Default (k=2)'].values

ax.bar(x - width/2, prepay_vals, width, label='Prepayment', color='steelblue', alpha=0.8)
ax.bar(x + width/2, default_vals, width, label='Default', color='indianred', alpha=0.8)

for i, (p, d) in enumerate(zip(prepay_vals, default_vals)):
    ax.text(i - width/2, p + 0.01, f'{p:.3f}', ha='center', fontsize=10)
    ax.text(i + width/2, d + 0.01, f'{d:.3f}', ha='center', fontsize=10)

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random (0.5)')
ax.set_xlabel('Time Horizon (months)')
ax.set_ylabel('Time-Dependent C-index')
ax.set_title('Breeden-Crook: Time-Dependent Concordance Index')
ax.set_xticks(x)
ax.set_xticklabels([f'tau = {h}' for h in horizons])
ax.set_ylim(0.4, 1.0)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'breeden_crook_cindex_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

The Breeden & Crook (2022) multihorizon model demonstrates the key finding that:
1. **Delinquency coefficients decay** with forecast horizon - their predictive power is strong for near-term predictions but diminishes over longer horizons
2. **Origination variables become more important** at longer horizons as delinquency's signal fades
3. **Model fit (Gini) decreases** with horizon, as near-future events are inherently easier to predict

This approach bridges the gap between:
- **Roll-rate/state-transition models** (accurate short-term via delinquency)
- **Vintage/APC models** (accurate long-term via origination quality)

### Next Steps
- Compare C-index results with DeepHit, Fine-Gray, and other models
- Run `python scripts/run_breeden_crook.py --max-horizon 12 --cif-horizon 72` on supercomputer